# Serve an STDIO MCP server over SSE with mcp-proxy

This notebook demonstrates how to expose an MCP server that communicates over STDIO (for example, `mcp-server-fetch`) as an MCP SSE endpoint using `mcp-proxy`.

[`mcp-proxy`](https://github.com/sparfenyuk/mcp-proxy) wraps any STDIO-based MCP server and exposes it as a proper MCP SSE transport endpoint, allowing clients to connect with `MCPServerSse` from the OpenAI Agents SDK  
instead of spawning a local STDIO subprocess per client. This makes it easy for multiple notebooks or agents to share a single server instance.

This lets multiple notebooks or agents connect to the same server concurrently.

What you'll learn
- Start `mcp-proxy` to wrap `mcp-server-fetch` and expose `/sse`.
- Connect from this notebook using `MCPServerSse` (OpenAI Agents SDK) and list available tools.
- Stop the proxy cleanly and manage the server process.
- Run `mcp-proxy` inside Docker as an alternative deployment.

Key advantages
- SSE uses standard HTTP connections, so the MCP server can run on a remote host and be reached over the network via its SSE endpoint.
- Server starts once and can be consumed by multiple notebooks/agents simultaneously.
- Clean async communication: MCP over SSE instead of multiple STDIO subprocesses.

Architecture
```
Notebook  ──MCPServerSse──►  mcp-proxy (:8000/sse)  ──stdio──►  mcp-server-fetch
```

## Standalone server (local mcp-proxy)

In this section we will run `mcp-proxy` locally to wrap `mcp-server-fetch`, verify the SSE endpoint is reachable,  
connect from this notebook using `MCPServerSse` to list available tools, and finally stop the proxy process.

Notes:

- Default port used in this notebook: `8000`.
- On Windows the launch command uses `cmd /c` and process control uses `PowerShell`.
- If the port is already in use, change the `MCP_PROXY_PORT` constant in the code below.

### 1. Start mcp-proxy (wrap `mcp-server-fetch`)

`mcp-proxy` launches `mcp-server-fetch` via STDIO and exposes it as an MCP SSE endpoint.

CLI (for reference):

```bash
uvx mcp-proxy --port 8000 uvx mcp-server-fetch
```

The SSE endpoint will be: `http://localhost:8000/sse`

In [ ]:
# Set params and commands based on the OS.
import sys

MCP_PROXY_PORT = 8000

# MCP Proxy launch command, uses the 'cmd /c' prefix version on Windows.
if sys.platform == 'win32':
    MCP_PROXY_CMD = f"cmd /c uvx mcp-proxy --port={MCP_PROXY_PORT} uvx mcp-server-fetch"
    GET_PID_CMD = f"powershell -Command Get-NetTCPConnection -LocalPort {MCP_PROXY_PORT} | Select-Object -ExpandProperty OwningProcess"
    KILL_PID_CMD = "powershell -Command Stop-Process -Id {} -Force"
else:  # Assume Unix-like. (Linux, macOS)
    MCP_PROXY_CMD = f"uvx mcp-proxy --port={MCP_PROXY_PORT} uvx mcp-server-fetch"
    GET_PID_CMD = f"lsof -i :{MCP_PROXY_PORT} -t"
    KILL_PID_CMD = "kill -9 {}"

In [ ]:
# Launch SSE to STDIO Proxy.
import subprocess

proxy_proc = subprocess.Popen(MCP_PROXY_CMD)

MCP_PROXY_SSE_URL = f"http://localhost:{MCP_PROXY_PORT}/sse"
print(f"mcp-proxy SSE endpoint: {MCP_PROXY_SSE_URL}")

In [ ]:
# Wait for the port to be open (socket check avoids hanging on the SSE stream)
import time
import socket

print("Waiting for mcp-proxy ", end='')
for _ in range(5):
    try:
        with socket.create_connection(("localhost", MCP_PROXY_PORT), timeout=1):
            print(" ✓ ready")
            break
    except OSError:
        print(f".", end='')
        time.sleep(0.5)
else:
    print(" ✗ failed to start")


### 2. Connect with `MCPServerSse` and list tools

Use `MCPServerSse` (OpenAI Agents SDK) to connect to the running MCP SSE endpoint and list the available tools.  
Functionally identical to `MCPServerStdio`, but using MCP over SSE instead of STDIO subprocesses.

In [ ]:
# Now we can connect to the MCP proxy server using the MCPServerSse client.
from agents.mcp import MCPServerSse

async with MCPServerSse(params={"url": MCP_PROXY_SSE_URL}, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

fetch_tools

### 3. Stop the proxy

Find the real server PID and terminate the MCP proxy process. The code cell below locates the server PID and stops it (Windows: PowerShell `Stop-Process`; Unix: `kill`).

In [ ]:
# Kill the server process on the real PID

# mcp-proxy process launched a detached server so we need to find the real PID
server_pid = subprocess.run(GET_PID_CMD, capture_output=True, text=True).stdout.strip()
subprocess.run(KILL_PID_CMD.format(server_pid))

## Docker (recommended)

Run `mcp-proxy` inside a container to avoid host process management and ensure a reproducible, isolated runtime.  
The container exposes the MCP SSE endpoint on a mapped host port so clients connect to `/sse` without launching local STDIO servers.

Benefits
- Isolated runtime and easy cleanup (start/stop containers instead of managing PIDs).
- Reproducible image across machines and CI.
- Simple port mapping to expose `/sse`.
- Easy to deploy and scale in a microservices architecture: agents on different hosts can share the same containerized service.
- More secure: container isolation reduces risk and makes it safer to run potentially dangerous tools (e.g., a REPL) inside the service.
- If the MCP server maintains shared state or "memory", that state is available to all agents connecting to the same endpoint.

Use the following cells to build the demo image, run the container (example: host 8001 → container 8000), connect with `MCPServerSse`, and stop the container.

### 1. Build Docker image

Build the demo image from the Dockerfile (next code cell runs `docker build`).

In [ ]:
import subprocess

image_name = "mcp-proxy-demo"
subprocess.run(f"docker build -t {image_name} .")

### 2. Launch container

Start the `mcp-proxy-demo` container and map a host port (example: host 8001 → container 8000).

In [ ]:
import subprocess

MCP_DOCKER_PORT = 8001
MCP_DOCKER_SSE_URL = f"http://localhost:{MCP_DOCKER_PORT}/sse"

subprocess.run(f"docker run -d --rm --name mcp-proxy-demo -p {MCP_DOCKER_PORT}:8000 mcp-proxy-demo")

In [ ]:
# Wait for the port to be open (socket check avoids hanging on the SSE stream)
import time
import socket

print("Waiting for mcp-proxy ", end='')
for _ in range(5):
    try:
        with socket.create_connection(("localhost", MCP_DOCKER_PORT), timeout=1):
            print(" ✓ ready")
            break
    except OSError:
        print(f".", end='')
        time.sleep(0.5)
else:
    print(" ✗ failed to start")

### 3. Connect with `MCPServerSse` (Docker)

Connect to the containerized SSE endpoint (`MCP_DOCKER_SSE_URL`) and list available tools. This mirrors the local flow but targets the container host and port.

In [ ]:
# Now we can connect to the MCP proxy server using the MCPServerSse client.
from agents.mcp import MCPServerSse

async with MCPServerSse(params={"url": MCP_DOCKER_SSE_URL}, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

fetch_tools

### 4. Stop server container

Stop and remove the `mcp-proxy-demo` container (example: `docker stop mcp-proxy-demo`).

In [ ]:
# Stop the Docker container for the MCP proxy demo server
import subprocess

subprocess.run("docker stop mcp-proxy-demo")